# 02 — Model training on the corrected 30-split registry

This stage trains the five fixed point-model specifications on the corrected cohort and the registered 30 cluster-disjoint splits prepared by `01_data_inventory.ipynb`. The reusable implementation is `scripts/train_model_splits.py`.

If an expected model file is absent or its stored feature specification differs from the registered 44-column docking panel, this notebook automatically starts or resumes training. Otherwise it reads and validates the persisted outputs without retraining.

The roles are fixed before evaluation: `train` fits the model and any preprocessing, `val` supplies split-conformal calibration residuals, and `test` is used only for reporting. The four registered exclusions are already absent from `data/df_final.csv`; no second filtering is applied.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODELS = ROOT / "results" / "models"
TABLES = ROOT / "results" / "tables"
MODEL_SLUGS = ("baseline", "pca", "adme", "plain", "bbb_pass")
EXPECTED_SPLIT_INDICES = range(30)
expected_model_files = [
    MODELS / f"split_{split_index:02d}" / model_slug / "model.cbm"
    for split_index in EXPECTED_SPLIT_INDICES
    for model_slug in MODEL_SLUGS
]
missing_model_files = [path for path in expected_model_files if not path.is_file()]
protocol_path = MODELS / "butina30_protocol.json"
previous_protocol = json.loads(protocol_path.read_text()) if protocol_path.is_file() else {}
needs_feature_spec_refresh = previous_protocol.get("raw_docking_feature_count") != 44
RUN_TRAINING = bool(missing_model_files) or needs_feature_spec_refresh

if RUN_TRAINING:
    reason = f"missing={len(missing_model_files)}, docking_feature_spec_refresh={needs_feature_spec_refresh}"
    print(f"Starting/resuming training because {reason}.")
    subprocess.run([sys.executable, str(ROOT / "scripts" / "train_model_splits.py"), "--resume"], check=True, cwd=ROOT)
else:
    print(f"All {len(expected_model_files)} models and the registered 44-column docking feature specification are present; reading saved artifacts only.")

assert not [path for path in expected_model_files if not path.is_file()], "Model training did not create every expected model.cbm file."
protocol = json.loads(protocol_path.read_text())
metrics = pd.read_csv(MODELS / "model_comparison_metrics.csv")
summary = pd.read_csv(MODELS / "model_comparison_summary.csv")
bbb_pairs = pd.read_csv(MODELS / "model_comparison_bbb_pair_audit.csv")

assert protocol["protocol"] == "butina_cluster_safe_random_splits_v2"
assert protocol["active_modeling_cohort"] == 12650
assert protocol["raw_docking_feature_count"] == 44
assert "6y1z" in protocol["raw_docking_features"]
assert protocol["excluded_ids_present_in_source_data"] == []
assert len(metrics) == 180
assert len(summary) == 6 and set(summary["n_splits"]) == {30}
assert len(bbb_pairs) == 30 and bbb_pairs["same_test_ids"].all()
print("Corrected 30-split point-model artifacts: PASS")

The table below is a descriptive summary across the 30 predefined splits. It is not yet a reviewer-facing model comparison: the paired differences and their confidence intervals required for Comment 1 are a separate subsequent analysis.

In [ ]:
display(summary[["model", "evaluation_domain", "n_splits", "RMSE_mean", "RMSE_std", "MAE_mean", "R2_mean", "Spearman_mean", "CI_0.90_size_mean", "CI_0.90_coverage_mean"]])

Persisted model files, calibration residuals, conformal quantiles, test predictions, and per-split metrics are under `results/models/split_00/` through `results/models/split_29/`. The three aggregate training CSV files are stored in `results/models/` alongside the generated model artifacts. No single-split Table 1 artifact is produced at this stage.

In [ ]:
# Artifact preview — show the saved cross-split model table after validation.
from IPython.display import Markdown, display

saved_summary_path = MODELS / "model_comparison_summary.csv"
assert saved_summary_path.is_file() and saved_summary_path.stat().st_size > 0, saved_summary_path
display(Markdown("## Saved 30-split model-summary table"))
display(pd.read_csv(saved_summary_path))
